# CE 310 — Week 11 Lab
## Hypothesis Testing: t-Tests and ANOVA

**Dataset:** `Week5_DOE_MedOffice_ZoneTemp_2023.csv` (Weeks 5–11 dataset)
**New tools:** `scipy.stats.ttest_1samp`, `scipy.stats.ttest_ind`, `scipy.stats.f_oneway`, `pairwise_tukeyhsd`

---
**Run all cells in order (Shift+Enter).** Fill every blank marked `___`.

## Before You Begin

**File needed:** `Week5_DOE_MedOffice_ZoneTemp_2023.csv`  
This is the same dataset from Weeks 5–7 (Excel) and Weeks 9–10 (Python).

**Upload options:**
1. **Colab Files panel (easiest):** Click the folder icon in the left sidebar → click the upload icon → select the CSV file from your computer.
2. **Google Drive mount:** Click the Drive mount button in the Files panel and navigate to the file.

> If you see `FileNotFoundError`, the CSV is not in Colab's working directory — go back and upload it first.

## Setup
Run the two cells below before anything else.

**New this week:** `scipy.stats` provides the three hypothesis-testing functions:
- `ttest_1samp(data, popmean)` — test whether a sample mean equals a known value
- `ttest_ind(group1, group2)` — compare means of two independent groups
- `f_oneway(*groups)` — one-way ANOVA: compare means of three or more groups

Both return a named tuple `(statistic, pvalue)` that you can unpack directly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.formula.api as smf

print('Libraries loaded.')

In [ ]:
# Upload Week5_DOE_MedOffice_ZoneTemp_2023.csv to Colab before running
df = pd.read_csv('Week5_DOE_MedOffice_ZoneTemp_2023.csv')
df['BH'] = (df['BusinessHours'] == 'Yes').astype(int)
print(f'Loaded {len(df):,} rows. Columns: {list(df.columns)}')
print(f'BH=1: {df["BH"].sum():,} occupied hours | BH=0: {(df["BH"]==0).sum():,} unoccupied hours')

---
## Section A — Hypothesis Testing Foundations

**Key ideas**

Every hypothesis test has:
1. **H₀ (null hypothesis):** the default assumption (e.g., "the two group means are equal")
2. **H₁ (alternative hypothesis):** what you're trying to show (e.g., "the means differ")
3. **Test statistic:** how far your data deviate from H₀ in standard-error units
4. **p-value:** probability of seeing a test statistic this extreme *if H₀ were true*
5. **Decision:** reject H₀ if p < α (typically α = 0.05)

You already know that occupied hours have much more cooling than unoccupied hours. This section formalises that observation as a statistical test.

### A1 — Within-Group Variability

Before testing whether two groups differ, we need to understand how spread out each group is.
A t-test divides the difference in means by the *standard error* of that difference —
which depends on each group's standard deviation and sample size.

Compute the standard deviation of `Cooling_kWh` separately for occupied and unoccupied hours,
rounded to 2 decimal places.

In [ ]:
# Separate the two groups
occ   = df[df['BH'] == 1]['Cooling_kWh']
unocc = df[df['BH'] == 0]['Cooling_kWh']

# Summary statistics
print(f'Occupied   (BH=1): n={len(occ):,}  mean={occ.mean():.2f}  median={occ.median():.2f}')
print(f'Unoccupied (BH=0): n={len(unocc):,}  mean={unocc.mean():.2f}  median={unocc.median():.2f}')
print()

ANSWER_A1_std_occ = round(___, 2)          # std of Cooling_kWh when BH=1
print(f'ANSWER_A1_std_occ = {ANSWER_A1_std_occ}')

### A2 — Two-Sample t-Test: Occupied vs. Unoccupied Cooling

**H₀:** Mean cooling during occupied hours = mean cooling during unoccupied hours  
**H₁:** The means are not equal (two-tailed test)  
**α = 0.05**

`scipy.stats.ttest_ind(group1, group2)` returns `(t_statistic, p_value)`.
A large |t| and small p (< 0.05) means we reject H₀.

In [ ]:
# Two-sample independent t-test
t_stat, p_val = stats.ttest_ind(___, ___)   # fill in the two group arrays

print(f't-statistic = {t_stat:.4f}')
print(f'p-value     = {p_val:.4e}')
print()

ANSWER_A2_t_stat = round(t_stat, 2)
print(f'ANSWER_A2_t_stat = {ANSWER_A2_t_stat}')

### A3 — One-Sample t-Test: Benchmark Comparison

A building energy benchmark for medium offices in hot climates suggests an average cooling
intensity of **20 kWh/hr** for all hours (occupied and unoccupied combined).

**H₀:** μ_cooling = 20 kWh/hr (this building matches the benchmark)  
**H₁:** μ_cooling ≠ 20 kWh/hr (this building differs from the benchmark)  

`scipy.stats.ttest_1samp(data, popmean)` tests whether the sample mean equals a stated value.

In [ ]:
# One-sample t-test against a benchmark of 20 kWh/hr
benchmark = 20.0
t_bench, p_bench = stats.ttest_1samp(___, benchmark)

print(f'Sample mean  = {df["Cooling_kWh"].mean():.4f} kWh/hr')
print(f'Benchmark    = {benchmark} kWh/hr')
print(f't-statistic  = {t_bench:.4f}')
print(f'p-value      = {p_bench:.4e}')
print()

ANSWER_A3_t_bench = round(t_bench, 2)
print(f'ANSWER_A3_t_bench = {ANSWER_A3_t_bench}')


*Type I and Type II errors (ungraded): Every hypothesis test can err in two ways. A **Type I error** (false positive, rate = α) means you reject H₀ when it is actually true — you conclude there is a difference when there is none. A **Type II error** (false negative, rate = β) means you fail to reject H₀ when it is actually false — you miss a real difference. In this lab, α = 0.05: we accept a 5% chance of a false positive. With n = 8,760 hours and a real effect, power (1 − β) is near 100% — we will almost certainly detect any true non-zero difference. The practical implication: in large datasets, statistically significant p-values are nearly guaranteed; what matters is effect size (Cohen's d).*


### A4 — One-Way ANOVA: Does Mean Cooling Differ by Season?

The t-test compares **two** groups. When there are **three or more** groups, use a
**one-way ANOVA** (Analysis of Variance).

**H₀:** μ_Fall = μ_Spring = μ_Summer = μ_Winter (all season means are equal)  
**H₁:** At least one season mean differs  

`scipy.stats.f_oneway(group1, group2, group3, group4)` returns `(F_statistic, p_value)`.
A large F-ratio (large between-group variance relative to within-group variance) and small p
means we reject H₀.

In [ ]:
# Separate data by season
fall   = df[df['Season'] == 'Fall']['Cooling_kWh']
spring = df[df['Season'] == 'Spring']['Cooling_kWh']
summer = df[df['Season'] == 'Summer']['Cooling_kWh']
winter = df[df['Season'] == 'Winter']['Cooling_kWh']

# One-way ANOVA
F_season, p_season = stats.f_oneway(___, ___, ___, ___)

print('Season means (kWh/hr):')
print(df.groupby('Season')['Cooling_kWh'].mean().round(2))
print()
print(f'F-statistic = {F_season:.4f}')
print(f'p-value     = {p_season:.4e}')
print()

ANSWER_A4_F_season = round(F_season, 2)
print(f'ANSWER_A4_F_season = {ANSWER_A4_F_season}')

### A5 — Effect Size: Cohen's d

A statistically significant p-value tells you the effect is **real** — but not whether it is
**large**. With n = 8,760, almost any true difference will be detectable.

**Cohen's d** measures effect size in standard-deviation units:

$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}$$

where the pooled standard deviation is:
$$s_{\text{pooled}} = \sqrt{\frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}}$$

Conventional benchmarks: |d| < 0.2 negligible · 0.2–0.5 small · 0.5–0.8 medium · > 0.8 large

In [ ]:
n1, s1 = len(occ), occ.std()
n2, s2 = len(unocc), unocc.std()

pooled_std = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
d = (occ.mean() - unocc.mean()) / pooled_std

print(f'Pooled std = {pooled_std:.4f} kWh/hr')
print(f"Cohen's d  = {d:.4f}")
print()

ANSWER_A5_cohens_d = round(d, 2)
print(f"ANSWER_A5_cohens_d = {ANSWER_A5_cohens_d}")

### A6 — Thermostat Compliance: Zone Temperature vs. 75°F Setpoint

ASHRAE Standard 55-2020 defines acceptable thermal comfort ranges. A common cooling setpoint
for office buildings is **75°F** during occupied hours. Let's test whether the building
actually maintains that setpoint across all 8,760 hours.

**H₀:** μ_zone = 75°F  
**H₁:** μ_zone ≠ 75°F

In [ ]:
# One-sample t-test: Zone_Temp_F vs. 75°F setpoint
setpoint = 75.0
t_zone, p_zone = stats.ttest_1samp(___, setpoint)

print(f'Mean Zone_Temp_F = {df["Zone_Temp_F"].mean():.4f} °F')
print(f'Setpoint         = {setpoint} °F')
print(f't-statistic      = {t_zone:.4f}')
print(f'p-value          = {p_zone:.4e}')
print()

ANSWER_A6_zone_t = round(t_zone, 2)
print(f'ANSWER_A6_zone_t = {ANSWER_A6_zone_t}')

### A7 — Seasonal Cooling Means

The ANOVA result from A4 tells us at least one season mean differs — but not *which* ones.
First, display the mean cooling for each season. Report the Summer mean to 2 decimal places.

In [ ]:
# Mean Cooling_kWh by season
season_means = df.groupby('Season')['Cooling_kWh'].mean().round(2)
print('Mean cooling by season (kWh/hr):')
print(season_means)
print()

ANSWER_A7_summer_cool = round(summer.mean(), 2)
print(f'ANSWER_A7_summer_cool = {ANSWER_A7_summer_cool}')

### A8 — Connection to Regression: t-Statistics in OLS

You may have noticed that the Model 2 summary from Week 10 included a `t` column. That column
shows the t-statistic for a hypothesis test on each coefficient:

**H₀:** coefficient = 0 (the predictor has no effect)  
**H₁:** coefficient ≠ 0  

The BH coefficient in Model 2 was 17.853 kWh/hr with a very small p-value. Re-extract its
t-statistic directly from the fitted model using `.tvalues['BH']`.

In [ ]:
m2 = smf.ols('Cooling_kWh ~ Outdoor_Temp_F + BH', data=df).fit()

bh_coef   = m2.params['BH']
bh_se     = m2.bse['BH']
bh_tval   = m2.tvalues['BH']

print(f'BH coefficient = {bh_coef:.4f} kWh/hr')
print(f'BH std error   = {bh_se:.4f}')
print(f'BH t-stat      = {bh_tval:.4f}   (coef / se = {bh_coef/bh_se:.4f})')
print(f'BH p-value     = {m2.pvalues["BH"]:.4e}')
print()

ANSWER_A8_BH_tstat = round(bh_tval, 2)
print(f'ANSWER_A8_BH_tstat = {ANSWER_A8_BH_tstat}')

---
## Section B — Extension Questions

**Working through Section B:** The theory cells explain the *why* — skim them
and focus on running the code. You do not need to read every word in class.

**B1** is a written question (manual grade) — answer it in the cell below.
**B2–B9** follow the same `ANSWER_` convention as Section A.

### B1 — Interpretation (Manual · 4 pts)

Using the results from A2 (two-sample t-test) and A5 (Cohen's d), write 3–5 sentences that:
1. State the H₀ and your conclusion (reject or fail to reject at α = 0.05).
2. Explain what the t-statistic of ≈ 55.7 tells you in plain language.
3. Argue whether the effect is *practically* significant, not just *statistically* significant,
   using the Cohen's d value and the actual mean difference in kWh/hr.

*Type your answer here:*

### B2 — Outdoor Temperature: Is the Confounding Statistically Significant?

In Week 10, you found that occupied hours occur at warmer outdoor temperatures on average
(75.9°F vs. 65.7°F). Is this 10°F gap statistically significant?

Run a two-sample t-test on `Outdoor_Temp_F` by BH group, and compute Cohen's d.

In [ ]:
occ_temp   = df[df['BH'] == 1]['Outdoor_Temp_F']
unocc_temp = df[df['BH'] == 0]['Outdoor_Temp_F']

t_out, p_out = stats.ttest_ind(___, ___)

# Cohen's d for outdoor temp
n1t, s1t = len(occ_temp), occ_temp.std()
n2t, s2t = len(unocc_temp), unocc_temp.std()
pool_t = np.sqrt(((n1t-1)*s1t**2 + (n2t-1)*s2t**2) / (n1t+n2t-2))
d_out  = (occ_temp.mean() - unocc_temp.mean()) / pool_t

print(f'Occupied   mean Outdoor_Temp_F = {occ_temp.mean():.2f} °F')
print(f'Unoccupied mean Outdoor_Temp_F = {unocc_temp.mean():.2f} °F')
print(f't-statistic = {t_out:.4f},  p-value = {p_out:.4e}')
print(f"Cohen's d   = {d_out:.4f}")
print()

ANSWER_B2_outdoor_t = round(t_out, 2)
ANSWER_B2_outdoor_d = round(d_out, 2)
print(f'ANSWER_B2_outdoor_t = {ANSWER_B2_outdoor_t}')
print(f'ANSWER_B2_outdoor_d = {ANSWER_B2_outdoor_d}')

### B3 — DayType: Weekday vs. Weekend Cooling

From Week 10, adding `DayType` (Weekday/Weekend) as a regression predictor alone explained
R² = 0.628 — less than BH (R² = 0.697). Test whether the difference in mean cooling between
Weekday and Weekend hours is statistically significant.

In [ ]:
weekday = df[df['DayType'] == 'Weekday']['Cooling_kWh']
weekend = df[df['DayType'] == 'Weekend']['Cooling_kWh']

t_dt, p_dt = stats.ttest_ind(___, ___)

n1d, s1d = len(weekday), weekday.std()
n2d, s2d = len(weekend), weekend.std()
pool_d  = np.sqrt(((n1d-1)*s1d**2 + (n2d-1)*s2d**2) / (n1d+n2d-2))
d_dt    = (weekday.mean() - weekend.mean()) / pool_d

print(f'Weekday mean = {weekday.mean():.2f} kWh/hr  (n={len(weekday):,})')
print(f'Weekend mean = {weekend.mean():.2f} kWh/hr  (n={len(weekend):,})')
print(f't-statistic  = {t_dt:.4f},  p-value = {p_dt:.4e}')
print(f"Cohen's d    = {d_dt:.4f}")
print()

ANSWER_B3_dt_t = round(t_dt, 2)
ANSWER_B3_dt_d = round(d_dt, 2)
print(f'ANSWER_B3_dt_t = {ANSWER_B3_dt_t}')
print(f'ANSWER_B3_dt_d = {ANSWER_B3_dt_d}')

### B4 — One-Way ANOVA by Month (12 Groups)

Season is a coarse grouping (4 levels). The monthly ANOVA uses 12 groups — one per calendar
month — giving a finer test of whether cooling varies systematically over the year.

Build the 12 month arrays, then pass them all to `f_oneway`.

In [ ]:
# Build one array per month
months = [df[df['Month'] == m]['Cooling_kWh'] for m in range(1, 13)]

# Monthly means for display
print('Monthly mean Cooling_kWh:')
print(df.groupby('Month')['Cooling_kWh'].mean().round(2).to_string())
print()

F_month, p_month = stats.f_oneway(___)
print(f'F-statistic = {F_month:.4f}')
print(f'p-value     = {p_month:.4e}')
print()

ANSWER_B4_F_month = round(F_month, 2)
print(f'ANSWER_B4_F_month = {ANSWER_B4_F_month}')

### B5 — Zone Temperature by Season

Report mean zone temperature for Summer and Winter (°F, rounded to 2 dp),
then run an ANOVA to test whether mean zone temperature differs by season.

In [ ]:
zt_means = df.groupby('Season')['Zone_Temp_F'].mean()
print('Mean Zone_Temp_F by season:')
print(zt_means.round(2))
print()

ANSWER_B5_zone_summer = round(zt_means['Summer'], 2)
ANSWER_B5_zone_winter = round(zt_means['Winter'], 2)
print(f'ANSWER_B5_zone_summer = {ANSWER_B5_zone_summer}')
print(f'ANSWER_B5_zone_winter = {ANSWER_B5_zone_winter}')

In [ ]:
# ANOVA: Zone_Temp_F by Season
zt_fall   = df[df['Season'] == 'Fall']['Zone_Temp_F']
zt_spring = df[df['Season'] == 'Spring']['Zone_Temp_F']
zt_summer = df[df['Season'] == 'Summer']['Zone_Temp_F']
zt_winter = df[df['Season'] == 'Winter']['Zone_Temp_F']

F_zone, p_zone = stats.f_oneway(___, ___, ___, ___)
print(f'Zone_Temp_F ANOVA by Season:')
print(f'F-statistic = {F_zone:.4f}')
print(f'p-value     = {p_zone:.4e}')
print()

ANSWER_B6_zone_F = round(F_zone, 2)
print(f'ANSWER_B6_zone_F = {ANSWER_B6_zone_F}')

### B7 — Summer vs. Winter: Direct Two-Sample t-Test

The Tukey HSD (cell below) tests all six season pairs simultaneously. Here, run a direct
two-sample t-test comparing **Summer** versus **Winter** cooling only.

In [ ]:
t_sw, p_sw = stats.ttest_ind(___, ___)

print(f'Summer cooling mean = {summer.mean():.2f} kWh/hr  (n={len(summer):,})')
print(f'Winter cooling mean = {winter.mean():.2f} kWh/hr  (n={len(winter):,})')
print(f't-statistic = {t_sw:.4f},  p-value = {p_sw:.4e}')
print()

ANSWER_B7_sw_t = round(t_sw, 2)
print(f'ANSWER_B7_sw_t = {ANSWER_B7_sw_t}')

### B8 — What Fraction of Unoccupied Hours Have Zero Cooling?

The unoccupied-hours median was 0 kWh/hr — meaning more than half of unoccupied hours
the system produces *no* cooling at all. Compute the exact percentage of unoccupied hours
where `Cooling_kWh == 0`, rounded to 1 decimal place.

In [ ]:
# Fraction with exactly zero cooling when unoccupied
zero_frac = (unocc == 0).mean()     # proportion [0, 1]
zero_pct  = round(zero_frac * 100, 1)

print(f'Unoccupied hours with Cooling_kWh = 0: {zero_pct}%')
print(f'  (out of {len(unocc):,} unoccupied hours)')
occ_zero = round((occ == 0).mean() * 100, 1)
print(f'Occupied hours with Cooling_kWh = 0:   {occ_zero}%')
print()

ANSWER_B8_zero_pct = zero_pct
print(f'ANSWER_B8_zero_pct = {ANSWER_B8_zero_pct}')


*Independence and autocorrelation (ungraded): The t-test and ANOVA assume observations are **independent**. Hourly building-energy data is not: the cooling load at 2 PM is correlated with the load at 1 PM (the building hasn't cooled down in one hour). This autocorrelation means the effective sample size n_eff < 8,760. A rough approximation: if lag-1 autocorrelation r₁ ≈ 0.9, then n_eff ≈ n × (1 − r₁)/(1 + r₁) ≈ 8,760 × 0.05 ≈ 438. Even with n_eff = 438, power is still very high for the large effects seen here (d > 1.0). The p-values will be optimistic (too small), but the qualitative conclusions remain valid.*


### B9 — Post-Hoc Analysis: Tukey HSD (Manual · 4 pts)

The ANOVA (A4) told us at least one season mean differs. The **Tukey Honestly Significant
Difference** test identifies *which* pairs of seasons differ, while controlling the
family-wise error rate at 5%.

Run the cell, then answer the question below.

In [ ]:
# Tukey HSD post-hoc test for Cooling_kWh by Season
tukey = pairwise_tukeyhsd(df['Cooling_kWh'], df['Season'], alpha=0.05)
print(tukey)

**B9 Written Response (4 pts):** Using the Tukey HSD table above, answer:
1. How many of the six season pairs are statistically significantly different at α = 0.05?
2. Which comparison has the largest mean difference, and by how much?
3. In one sentence, explain why Summer–Winter has the largest difference but
   the p-value for every pair is listed as exactly 0.001.

*Type your answer here:*

### B10 — Written Reflection: Statistical vs. Practical Significance (Manual · 4 pts)

Consider the following three results from this lab:

| Test | t or F | p-value | Cohen's d |
|---|---|---|---|
| Cooling: BH=1 vs BH=0 | t = 55.68 | < 0.001 | d = 1.30 |
| Cooling: Weekday vs Weekend | t = 16.01 | < 0.001 | d = 0.38 |
| Outdoor Temp: BH=1 vs BH=0 | t = 26.76 | < 0.001 | d = 0.63 |

In 4–6 sentences, explain:
1. Why all three tests have p < 0.001 even though the effect sizes vary widely.
2. Which test represents the most *practically* significant result for an engineer
   sizing HVAC equipment, and why?
3. When would you report Cohen's d instead of (or in addition to) a p-value?

*Type your answer here:*

---
## Memo (Manual · 8 pts)

Write a **3-paragraph technical memo** to a building owner summarising what the
hypothesis tests from this lab reveal about the building's cooling system.
Use specific numbers from your own results — do not copy them from the prompts below.

**Paragraph 1 — Occupancy effect (3 pts)**  
Cover: whether occupied and unoccupied cooling loads are statistically different,
what the t-statistic and p-value tell a non-statistician, the mean difference
in kWh/hr, and what Cohen's d says about the practical magnitude of the effect.

**Paragraph 2 — Seasonal variation (3 pts)**  
Cover: whether season significantly affects cooling demand (cite the F-statistic),
which season has the highest and lowest mean cooling load, and how this connects
to the regression findings from Weeks 9–10.

**Paragraph 3 — Thermostat compliance (2 pts)**  
Cover: whether mean zone temperature is significantly different from the 75°F setpoint
(cite your t-statistic), whether the deviation is practically meaningful, and
what this implies for occupant comfort and HVAC sizing.

*Type memo here:*

---
## Looking Ahead: Week 12 — Sensitivity Analysis

This week you tested whether observed differences are *statistically real*.
Next week you will ask: if the *inputs to your model* are uncertain, how much uncertainty
does that create in the *outputs*?

**Sensitivity analysis** quantifies how much each input variable drives variation in an
output — a key tool for engineering design under uncertainty. You will apply it to the
regression model from Weeks 9–10 to assess which input (outdoor temperature, occupancy
schedule, or season) contributes most to uncertainty in annual cooling energy predictions.

---
## Before You Submit

> ⚠️ Your saved cell **outputs** are what get read, not your code. If you skip this step, every numeric question scores 0 — even if your formulas are correct.

**Step 1 — Run All Cells:**
In the menu bar: **Runtime → Run all** (Google Colab) or **Kernel → Restart & Run All** (Jupyter).
Wait until every cell shows output — no spinning circles.

**Step 2 — Check your ANSWER_ lines:**
Each graded cell should print a line like `ANSWER_A1 = 23.45`. If any prints `None` or is missing, fix the code and re-run.

**Step 3 — Download and upload:**
- File → Download → **Download .ipynb** (not PDF, not .py)
- Rename: `lastname_firstname_week11.ipynb`
- Upload to D2L before the deadline

**Submission checklist:**
- [ ] All cells ran without error (no red tracebacks)
- [ ] Every `ANSWER_` variable prints a number (not `None`)
- [ ] Memo paragraphs are filled in (not placeholder text)
- [ ] File downloaded as `.ipynb` — not PDF, not `.py`
- [ ] File renamed: `lastname_firstname_week11.ipynb`
